In [1]:
import sys
print(sys.executable)

/usr/bin/python3


In [2]:
#!pip install torch==2.9.0+cu129 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129

In [3]:
#!pip install pytorch-lightning 

In [4]:
!nvidia-smi

Wed Nov 12 01:01:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   42C    P8             12W /  600W |      41MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
#!pip install transformers datasets pycocotools accelerate pillow timm tensorboard scipy

In [6]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [7]:
%load_ext tensorboard
%tensorboard --logdir ./logs

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

import torch
import warnings
import os
from PIL import Image
from pycocotools.coco import COCO

# Importações do Hugging Face e PyTorch
from transformers import (
    DeformableDetrImageProcessor,
    DeformableDetrForObjectDetection,
    TrainingArguments,
    Trainer,
)
from torch.utils.data import DataLoader
from torchvision.datasets import CocoDetection # <--- Importação chave!

# ⚠️ Ignorar warnings (como os de precisão mista ou de compatibilidade)
warnings.filterwarnings(
    "ignore",
    message=".*copying from a non-meta parameter.*",
    category=UserWarning,
    module="torch.nn.modules.module"
)
warnings.filterwarnings(
    "ignore",
    message=".*The given NumPy array is not writeable.*",
    category=UserWarning,
    module="pycocotools.coco"
)


# ============================================================
# 1️⃣ Variáveis e Configuração do Ambiente
# ============================================================
# Definição do Device para a GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device detectado: {device}")

# Caminhos do dataset
ROOT_DIR = "/home/usr/SVRDD_COCO"
TRAIN_DIRECTORY = os.path.join(ROOT_DIR, "train")
VAL_DIRECTORY = os.path.join(ROOT_DIR, "valid")
ANNOTATION_FILE_NAME = "_annotations.coco.json" # Nome padrão do seu arquivo

# Labels
id2label = {
    0: "background", 1: "longitudinal crack", 2: "transverse crack", 3: "alligator crack",
    4: "pothole", 5: "manhole cover", 6: "longitudinal patch",
    7: "transverse patch",
}
label2id = {v: k for k, v in id2label.items()}

# Modelo base
model_name = "SenseTime/deformable-detr"


# ============================================================
# 2️⃣ Classe Customizada para Dataloading Robusto
# ============================================================
class CocoDetectionCustom(CocoDetection):
    # Herda a lógica de leitura de arquivos COCO do torchvision
    def __init__(self, image_directory_path: str, image_processor):
        annotation_file_path = os.path.join(image_directory_path, ANNOTATION_FILE_NAME)
        
        # O __init__ do CocoDetection carrega as anotações e os caminhos
        super(CocoDetectionCustom, self).__init__(image_directory_path, annotation_file_path)
        
        self.image_processor = image_processor

    def __getitem__(self, idx):
        # 1. Obter Imagem (PIL Image) e Anotações (lista COCO)
        image, annotations = super(CocoDetectionCustom, self).__getitem__(idx)
        image_id = self.ids[idx] # Obtém o ID da imagem do índice

        # 🛑 FILTRAGEM DE ANOTAÇÕES QUEBRADAS (CRUCIAL!) 🛑
        MIN_DIM = 2.0  # Mínimo de 2 pixels (no tamanho original 1024)
                       # Pode ajustar se 2.0 ainda for muito pequeno
        
        annotations = [
            ann for ann in annotations 
            # Verifica área
            if ann["area"] > 0
            # Verifica largura (bbox[2]) e altura (bbox[3])
            and ann["bbox"][2] > MIN_DIM 
            and ann["bbox"][3] > MIN_DIM
        ]

        
        
        # 2. Formatar anotações para o processor do Hugging Face
        annotations = {'image_id': image_id, 'annotations': annotations}
        
        # 3. Processar a amostra (redimensionamento e conversão COCO -> DETR)
        # return_tensors="pt" é aplicado aqui, mas sem padding
        encoding = self.image_processor(images=image, annotations=annotations, return_tensors="pt")
        
        # 4. Retornar amostra sem a dimensão de batch (squeeze)
        return {
            "pixel_values": encoding["pixel_values"].squeeze(),
            "pixel_mask": encoding["pixel_mask"].squeeze(),
            "labels": encoding["labels"][0], # O labels é uma lista, pegamos o primeiro elemento
        }

# ============================================================
# 3️⃣ Inicialização do Modelo e Datasets
# ============================================================
print("Carregando modelo e processor...")
processor = DeformableDetrImageProcessor.from_pretrained(model_name)

print("Instanciando Datasets...")
TRAIN_DATASET = CocoDetectionCustom(
    image_directory_path=TRAIN_DIRECTORY, 
    image_processor=processor
)
VAL_DATASET = CocoDetectionCustom(
    image_directory_path=VAL_DIRECTORY, 
    image_processor=processor
)
TEST_DATASET = CocoDetectionCustom(
    image_directory_path=VAL_DIRECTORY, 
    image_processor=processor
)

# ============================================================
# 4️⃣ Collate Function para BATCHING e PADDING
# ============================================================
def collate_fn(batch):
    pixel_values = [item["pixel_values"] for item in batch]
    encoding = processor.pad(pixel_values, return_tensors="pt")
    labels = [item["labels"] for item in batch]
    return {
        'pixel_values': encoding['pixel_values'],
        'pixel_mask': encoding['pixel_mask'],
        'labels': labels
    }
TRAIN_DATALOADER = DataLoader(dataset=TRAIN_DATASET, collate_fn=collate_fn, batch_size=4, shuffle=True)
VAL_DATALOADER = DataLoader(dataset=VAL_DATASET, collate_fn=collate_fn, batch_size=4)
TEST_DATALOADER = DataLoader(dataset=TEST_DATASET, collate_fn=collate_fn, batch_size=4)

Device detectado: cuda:0
Carregando modelo e processor...
Instanciando Datasets...
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [20]:
from transformers import DeformableDetrForObjectDetection
import torch
import pytorch_lightning as pl

class DeformableDetr(pl.LightningModule): # Mude o nome da classe para clareza
    def __init__(self, lr, lr_backbone, weight_decay):
        super().__init__()
        
        self.model = DeformableDetrForObjectDetection.from_pretrained(
            pretrained_model_name_or_path=model_name, 
            num_labels=len(id2label),
            ignore_mismatched_sizes=True
        )
        
        self.lr = lr
        self.lr_backbone = lr_backbone
        self.weight_decay = weight_decay

    def forward(self, pixel_values, pixel_mask):
        return self.model(pixel_values=pixel_values, pixel_mask=pixel_mask)

    def common_step(self, batch, batch_idx):
        pixel_values = batch["pixel_values"]
        pixel_mask = batch["pixel_mask"]
        print("batch['labels'] =", batch['labels'])
        labels = [{k: v.to(self.device) for k, v in t.items()} for t in batch["labels"]]

        outputs = self.model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)

        loss = outputs.loss
        loss_dict = outputs.loss_dict

        return loss, loss_dict

    def training_step(self, batch, batch_idx):
        loss, loss_dict = self.common_step(batch, batch_idx)     
        # logs metrics for each training_step, and the average across the epoch
        self.log("training_loss", loss)
        for k,v in loss_dict.items():
            self.log("train_" + k, v.item())

        return loss
 
    def validation_step(self, batch, batch_idx):
        loss, loss_dict = self.common_step(batch, batch_idx)     
        self.log("validation/loss", loss)
        for k, v in loss_dict.items():
            self.log("validation_" + k, v.item())
            
        return loss

    def configure_optimizers(self):
        # DETR authors decided to use different learning rate for backbone
        # you can learn more about it here: 
        # - https://github.com/facebookresearch/detr/blob/3af9fa878e73b6894ce3596450a8d9b89d918ca9/main.py#L22-L23
        # - https://github.com/facebookresearch/detr/blob/3af9fa878e73b6894ce3596450a8d9b89d918ca9/main.py#L131-L139
        param_dicts = [
            {
                "params": [p for n, p in self.named_parameters() if "backbone" not in n and p.requires_grad]},
            {
                "params": [p for n, p in self.named_parameters() if "backbone" in n and p.requires_grad],
                "lr": self.lr_backbone,
            },
        ]
        return torch.optim.AdamW(param_dicts, lr=self.lr, weight_decay=self.weight_decay)

    def train_dataloader(self):
        return TRAIN_DATALOADER

    def val_dataloader(self):
        return VAL_DATALOADER

In [21]:
# ============================================================
# 5️⃣ TrainingArguments (Estável e Otimizado)
# ============================================================

model = DeformableDetr(lr=1e-5, lr_backbone=1e-5, weight_decay=1e-4)

batch = next(iter(TRAIN_DATALOADER))
outputs = model(pixel_values=batch['pixel_values'], pixel_mask=batch['pixel_mask'])

Some weights of the model checkpoint at SenseTime/deformable-detr were not used when initializing DeformableDetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DeformableDetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DeformableDetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DeformableDetrForObjectDetection wer

In [22]:
from pytorch_lightning import Trainer

#args = TrainingArguments(
#    output_dir="./deformable-detr-finetuned-asphalt",
#    per_device_train_batch_size=8,   # Uso eficiente da 5090
#    per_device_eval_batch_size=4,
#    gradient_accumulation_steps=1,   
#    learning_rate=1e-5,              # 🚨 LR de segurança (aumentar após estabilidade)
#    warmup_steps=1000,               # ✅ Essencial para estabilidade
#    num_train_epochs=50,             # Número razoável de épocas
#    logging_dir="./logs",
#    logging_steps=100,
#    report_to="tensorboard",
#    disable_tqdm=False,
#    save_strategy="epoch",
#    eval_strategy="epoch",
#    fp16=False,                      # 🛑 FP32 para máxima estabilidade inicial
#    remove_unused_columns=False,
#    dataloader_pin_memory=True,      # Otimização de I/O
#)

# ============================================================
# 6️⃣ Trainer e Início
# ============================================================
trainer = Trainer(devices=1, accelerator="gpu", max_epochs=100, gradient_clip_val=1.0, accumulate_grad_batches=8, log_every_n_steps=5, precision=32)


print("🚀 Iniciando treinamento com Pytorch Trainer...")
trainer.fit(model)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type                             | Params | Mode
------------------------------------------------------------------
0 | model | DeformableDetrForObjectDetection | 40.0 M | eval
------------------------------------------------------------------
39.8 M    Trainable params
222 K     Non-trainable params
40.0 M    Total params
160.193   Total estimated model params size (MB)
0         Modules in train mode
425       Modules in eval mode


🚀 Iniciando treinamento com Pytorch Trainer...


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

batch['labels'] = [{'size': tensor([800, 800], device='cuda:0'), 'image_id': tensor([1], device='cuda:0'), 'class_labels': tensor([2], device='cuda:0'), 'boxes': tensor([[0.5669, 0.6445, 0.4229, 0.0215]], device='cuda:0'), 'area': tensor([5814.1138], device='cuda:0'), 'iscrowd': tensor([0], device='cuda:0'), 'orig_size': tensor([1024, 1024], device='cuda:0')}, {'size': tensor([800, 800], device='cuda:0'), 'image_id': tensor([2], device='cuda:0'), 'class_labels': tensor([2], device='cuda:0'), 'boxes': tensor([[0.4541, 0.6763, 0.9062, 0.0615]], device='cuda:0'), 'area': tensor([35683.3398], device='cuda:0'), 'iscrowd': tensor([0], device='cuda:0'), 'orig_size': tensor([1024, 1024], device='cuda:0')}, {'size': tensor([800, 800], device='cuda:0'), 'image_id': tensor([3], device='cuda:0'), 'class_labels': tensor([7], device='cuda:0'), 'boxes': tensor([[0.2080, 0.6724, 0.4141, 0.0869]], device='cuda:0'), 'area': tensor([23032.1816], device='cuda:0'), 'iscrowd': tensor([0], device='cuda:0'), 

ValueError: matrix contains invalid numeric entries

In [ ]:
from datasets import Dataset
from pycocotools.coco import COCO
from PIL import Image
import os
import transformers
from transformers import DeformableDetrImageProcessor, DeformableDetrForObjectDetection, TrainingArguments, Trainer
import torch
import warnings 

warnings.filterwarnings(
    "ignore",
    message=".*copying from a non-meta parameter.*",
    category=UserWarning,
    module="torch.nn.modules.module"
)

# 1️⃣ Caminhos do dataset COCO
train_img_dir = "/home/usr/SVRDD_COCO/train"
val_img_dir = "/home/usr/SVRDD_COCO/valid"
test_img_dir = "/home/usr/SVRDD_COCO/test"

train_path = f"{train_img_dir}/_annotations.coco.json"
val_path = f"{val_img_dir}/_annotations.coco.json"

# 2️⃣ Definições do dataset
id2label = {
    0: "longitudinal crack",
    1: "transverse crack",
    2: "alligator crack",
    3: "pothole",
    4: "manhole cover",
    5: "longitudinal patch",
    6: "transverse patch",
}
label2id = {v: k for k, v in id2label.items()}

# 3️⃣ Função para converter COCO → HuggingFace Dataset
def coco_to_list(coco, image_dir):
    imgs = coco.loadImgs(coco.getImgIds())
    dataset = []
    for img in imgs:
        anns = coco.loadAnns(coco.getAnnIds(imgIds=img["id"]))
        objects = []
        for ann in anns:
            objects.append({
                "bbox": ann["bbox"],
                "category_id": ann["category_id"] - 1,
                "area": ann["area"],
                "iscrowd": ann.get("iscrowd", 0)
            })
        dataset.append({
            "image_id": img["id"],
            "image_path": os.path.join(image_dir, img["file_name"]),
            "objects": objects,
        })
    return dataset

print("Carregando dataset COCO...")
train_coco = COCO(train_path)
val_coco = COCO(val_path)

print("Convertendo para Dataset HuggingFace...")
train_dataset = Dataset.from_list(coco_to_list(train_coco, train_img_dir))
val_dataset = Dataset.from_list(coco_to_list(val_coco, val_img_dir))


print("Carregando modelo e processor...")
model_name = "SenseTime/deformable-detr"
processor = DeformableDetrImageProcessor.from_pretrained(model_name, do_convert_annotations=True)
model = DeformableDetrForObjectDetection.from_pretrained(
    model_name,
    num_labels=len(id2label),
    ignore_mismatched_sizes=True,
    id2label=id2label,
    label2id=label2id,
)


# 5️⃣ Collate function (CORRIGIDA)
def collate_fn(batch):
    # O 'batch' é uma lista de dicionários com dados brutos (image_path, objects)
    
    # 1. Carregar Imagens
    images = [Image.open(example["image_path"]).convert("RGB") for example in batch]
    
    # 2. Formatar Anotações para o Processor
    annotations = [
        {"image_id": example["image_id"], "annotations": example["objects"]}
        for example in batch
    ]
    
    # 3. Processar o BATCH COMPLETO (o processor lida com normalização e padding)
    # O processor retornará o dicionário {pixel_values, pixel_mask, labels}
    inputs = processor(images=images, annotations=annotations, return_tensors="pt")
    
    # 4. Mover TUDO para a GPU antes de retornar ao Trainer
    # O Trainer se encarrega disso se o modelo estiver na GPU, mas é mais seguro.
    inputs["pixel_values"] = inputs["pixel_values"].to(device)
    inputs["pixel_mask"] = inputs["pixel_mask"].to(device)
    
    # Mover as labels (que são listas de dicionários) manualmente
    labels_on_device = []
    for label_dict in inputs["labels"]:
        new_label_dict = {}
        for k, v in label_dict.items():
            if torch.is_tensor(v):
                new_label_dict[k] = v.to(device)
            else:
                new_label_dict[k] = v
        labels_on_device.append(new_label_dict)
        
    inputs["labels"] = labels_on_device
    
    return inputs

    

# 6️⃣ TrainingArguments
args = TrainingArguments(
    output_dir="./deformable-detr-finetuned-asphalt",
    per_device_train_batch_size=8,  # pelo menos 2 por GPU
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,  # simula batch 4 se necessário
    learning_rate=1e-5,
    num_train_epochs=100,
    logging_dir="./logs",
    logging_steps=100,
    report_to="tensorboard",  # só printa no console
    disable_tqdm=False,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=False,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

# 7️⃣ Trainer com accelerate
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)


print("Iniciando treinamento...")
trainer.train()

In [ ]:
!export CUDA_LAUNCH_BLOCKING=1

from datasets import Dataset
from pycocotools.coco import COCO
from PIL import Image
import os
import transformers
from transformers import (
    DeformableDetrImageProcessor,
    DeformableDetrForObjectDetection,
    TrainingArguments,
    Trainer,
)
import torch
import warnings 

# ⚠️ Ignorar warnings irrelevantes do PyTorch
warnings.filterwarnings(
    "ignore",
    message=".*copying from a non-meta parameter.*",
    category=UserWarning,
    module="torch.nn.modules.module"
)

# ⚙️ Ativar autotuner da cuDNN (melhora desempenho em convoluções)
#torch.backends.cudnn.benchmark = True

# ============================================================
# 1️⃣ Caminhos do dataset COCO
# ============================================================
train_img_dir = "/home/usr/SVRDD_COCO/train"
val_img_dir = "/home/usr/SVRDD_COCO/valid"
test_img_dir = "/home/usr/SVRDD_COCO/test"

train_path = f"{train_img_dir}/_annotations.coco.json"
val_path = f"{val_img_dir}/_annotations.coco.json"

# ============================================================
# 2️⃣ Labels
# ============================================================
id2label = {
    0: "longitudinal crack",
    1: "transverse crack",
    2: "alligator crack",
    3: "pothole",
    4: "manhole cover",
    5: "longitudinal patch",
    6: "transverse patch",
}
label2id = {v: k for k, v in id2label.items()}

# ============================================================
# 3️⃣ Conversão COCO → Dataset
# ============================================================
def coco_to_list(coco, image_dir):
    imgs = coco.loadImgs(coco.getImgIds())
    dataset = []
    for img in imgs:
        anns = coco.loadAnns(coco.getAnnIds(imgIds=img["id"]))
        objects = []
        for ann in anns:
            objects.append({
                "bbox": ann["bbox"],
                "category_id": ann["category_id"] - 1,
                "area": ann["area"],
                "iscrowd": ann.get("iscrowd", 0)
            })
        dataset.append({
            "image_id": img["id"],
            "image_path": os.path.join(image_dir, img["file_name"]),
            "objects": objects,
        })
    return dataset

print("Carregando dataset COCO...")
train_coco = COCO(train_path)
val_coco = COCO(val_path)

print("Convertendo para Dataset...")
train_dataset = Dataset.from_list(coco_to_list(train_coco, train_img_dir))
val_dataset = Dataset.from_list(coco_to_list(val_coco, val_img_dir))
print(f"Tamanhos: train={len(train_dataset)}, val={len(val_dataset)}")

# ============================================================
# 4️⃣ Modelo e Processor
# ============================================================
model_name = "SenseTime/deformable-detr"
processor = DeformableDetrImageProcessor.from_pretrained(model_name, do_convert_annotations=True)
model = DeformableDetrForObjectDetection.from_pretrained(
    model_name,
    num_labels=len(id2label),
    ignore_mismatched_sizes=True,
    id2label=id2label,
    label2id=label2id,
)

def collate_fn(batch):
    # O 'batch' aqui é uma lista de dicionários, onde cada dicionário
    # contém os dados brutos de uma imagem (image_path, objects, etc.)
    
    # 1. Extrair imagens e anotações do batch de dados brutos
    images = [Image.open(example["image_path"]).convert("RGB") for example in batch]
    
    # A estrutura de anotações deve ser no formato COCO (lista de dicionários)
    annotations = [
        {"image_id": example["image_id"], "annotations": example["objects"]}
        for example in batch
    ]
    
    # 2. Chamar o processor no BATCH COMPLETO.
    # O processor lida com:
    # a) Redimensionamento/Normalização.
    # b) Conversão das anotações COCO para o formato DETR (boxes normalizadas, classes).
    # c) PADDING e empilhamento dos tensores (pixel_values e pixel_mask).
    # d) Retorno dos tensores PyTorch prontos para o modelo.
    inputs = processor(images=images, annotations=annotations, return_tensors="pt")
    
    # inputs agora é um dicionário {pixel_values, pixel_mask, labels} prontos.
    return inputs


# ============================================================
# 6️⃣ Argumentos de treino — versão otimizada para 5090
# ============================================================
args = TrainingArguments(
    output_dir="./deformable-detr-finetuned-asphalt",
    per_device_train_batch_size=8,       # 🔼 DOBRADO
    per_device_eval_batch_size=2,
    learning_rate=1e-5,
    num_train_epochs=50,
    #weight_decay=1e-4,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=3,
    remove_unused_columns=False,
    fp16=False,                           # ⚡ ATIVAR MIXED PRECISION
    gradient_accumulation_steps=1,       # Pode aumentar se quiser batch lógico maior
    # Paraleliza o carregamento
    torch_compile=False,                  # 🧠 Ativa compilação JIT do PyTorch 2.x (grande boost)
    dataloader_pin_memory=False,
    disable_tqdm=False,
    warmup_steps=1000,
)

# ============================================================
# 7️⃣ Trainer
# ============================================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# ============================================================
# 8️⃣ Treinar!
# ============================================================
print("🚀 Iniciando treino otimizado na RTX 5090...")
trainer.train()


In [ ]:
torch.cuda.empty_cache()

In [ ]:
print(torch.version.cuda)       # Versão do CUDA que o PyTorch está usando
print(torch.backends.cudnn.version())  # Versão do cuDNN
print(torch.cuda.is_available()) 

In [ ]:
import os 
import torch
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.device_count():", torch.cuda.device_count())
print(next(model.parameters()).device)

In [ ]:
!export CUDA_LAUNCH_BLOCKING=1

from datasets import Dataset
from pycocotools.coco import COCO
from PIL import Image
import os
import transformers
from transformers import (
    DeformableDetrImageProcessor,
    DeformableDetrForObjectDetection,
    TrainingArguments,
    Trainer,
)
import torch
import warnings 

# ⚠️ Ignorar warnings irrelevantes do PyTorch
warnings.filterwarnings(
    "ignore",
    message=".*copying from a non-meta parameter.*",
    category=UserWarning,
    module="torch.nn.modules.module"
)

# ⚙️ Ativar autotuner da cuDNN (melhora desempenho em convoluções)
#torch.backends.cudnn.benchmark = True

# ============================================================
# 1️⃣ Caminhos do dataset COCO
# ============================================================
train_img_dir = "/home/usr/SVRDD_COCO/train"
val_img_dir = "/home/usr/SVRDD_COCO/valid"
test_img_dir = "/home/usr/SVRDD_COCO/test"

train_path = f"{train_img_dir}/_annotations.coco.json"
val_path = f"{val_img_dir}/_annotations.coco.json"

# ============================================================
# 2️⃣ Labels
# ============================================================
id2label = {
    0: "longitudinal crack",
    1: "transverse crack",
    2: "alligator crack",
    3: "pothole",
    4: "manhole cover",
    5: "longitudinal patch",
    6: "transverse patch",
}
label2id = {v: k for k, v in id2label.items()}

# ============================================================
# 3️⃣ Conversão COCO → Dataset
# ============================================================
def coco_to_list(coco, image_dir):
    imgs = coco.loadImgs(coco.getImgIds())
    dataset = []
    for img in imgs:
        anns = coco.loadAnns(coco.getAnnIds(imgIds=img["id"]))
        objects = []
        for ann in anns:
            objects.append({
                "bbox": ann["bbox"],
                "category_id": ann["category_id"] - 1,
                "area": ann["area"],
                "iscrowd": ann.get("iscrowd", 0)
            })
        dataset.append({
            "image_id": img["id"],
            "image_path": os.path.join(image_dir, img["file_name"]),
            "objects": objects,
        })
    return dataset

print("Carregando dataset COCO...")
train_coco = COCO(train_path)
val_coco = COCO(val_path)

print("Convertendo para Dataset...")
train_dataset = Dataset.from_list(coco_to_list(train_coco, train_img_dir))
val_dataset = Dataset.from_list(coco_to_list(val_coco, val_img_dir))
print(f"Tamanhos: train={len(train_dataset)}, val={len(val_dataset)}")

# ============================================================
# 4️⃣ Modelo e Processor
# ============================================================
model_name = "SenseTime/deformable-detr"
processor = DeformableDetrImageProcessor.from_pretrained(model_name, do_convert_annotations=True)
model = DeformableDetrForObjectDetection.from_pretrained(
    model_name,
    num_labels=len(id2label),
    ignore_mismatched_sizes=True,
    id2label=id2label,
    label2id=label2id,
)

# ============================================================
# 5️⃣ Transform e Collate
# ============================================================
def transform(example_batch):
    # 1. Carregar imagens e anotações (igual ao seu código)
    images = [Image.open(p).convert("RGB") for p in example_batch["image_path"]]
    annotations = [
        {"image_id": id, "annotations": ann}
        for id, ann in zip(example_batch["image_id"], example_batch["objects"])
    ]
    
    # 2. Processar a amostra
    # return_tensors="pt" é aplicado aqui para obter tensores PyTorch
    inputs = processor(images=images, annotations=annotations, return_tensors="pt")
    
    # 3. Retornar os tensores como lista/dicionário para o Datasets
    # O datasets.map espera um dicionário.
    # Usamos .squeeze(0) porque o processor retorna (1, C, H, W), e queremos (C, H, W)
    # para que o processor.pad possa fazer o batching no collate_fn.
    return {
        "pixel_values": inputs["pixel_values"].squeeze(0),
        "pixel_mask": inputs["pixel_mask"].squeeze(0),
        "labels": inputs["labels"][0], # as labels já estão no formato de lista de dicionários
    }
#train_dataset = train_dataset.with_transform(transform)
#val_dataset = val_dataset.with_transform(transform)

#def collate_fn(batch):
#    pixel_values = torch.stack([b["pixel_values"] for b in batch])
#    labels = [b["labels"] for b in batch]
#    return {"pixel_values": pixel_values, "labels": labels}


#def collate_fn(batch):
#    return processor.pad(batch, return_tensors="pt")

def collate_fn(batch):
    # O 'batch' aqui é uma lista de dicionários, onde cada dicionário
    # contém os dados brutos de uma imagem (image_path, objects, etc.)
    
    # 1. Extrair imagens e anotações do batch de dados brutos
    images = [Image.open(example["image_path"]).convert("RGB") for example in batch]
    
    # A estrutura de anotações deve ser no formato COCO (lista de dicionários)
    annotations = [
        {"image_id": example["image_id"], "annotations": example["objects"]}
        for example in batch
    ]
    
    # 2. Chamar o processor no BATCH COMPLETO.
    # O processor lida com:
    # a) Redimensionamento/Normalização.
    # b) Conversão das anotações COCO para o formato DETR (boxes normalizadas, classes).
    # c) PADDING e empilhamento dos tensores (pixel_values e pixel_mask).
    # d) Retorno dos tensores PyTorch prontos para o modelo.
    inputs = processor(images=images, annotations=annotations, return_tensors="pt")
    
    # inputs agora é um dicionário {pixel_values, pixel_mask, labels} prontos.
    return inputs


# ============================================================
# 6️⃣ Argumentos de treino — versão otimizada para 5090
# ============================================================
args = TrainingArguments(
    output_dir="./deformable-detr-finetuned-asphalt",
    per_device_train_batch_size=8,       # 🔼 DOBRADO
    per_device_eval_batch_size=2,
    learning_rate=1e-5,
    num_train_epochs=200,
    #weight_decay=1e-4,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=3,
    remove_unused_columns=False,
    fp16=False,                           # ⚡ ATIVAR MIXED PRECISION
    gradient_accumulation_steps=1,       # Pode aumentar se quiser batch lógico maior
    # Paraleliza o carregamento
    torch_compile=False,                  # 🧠 Ativa compilação JIT do PyTorch 2.x (grande boost)
    dataloader_pin_memory=False,
    disable_tqdm=False,
)

# ============================================================
# 7️⃣ Trainer
# ============================================================
#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#model.to(device)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# ============================================================
# 8️⃣ Treinar!
# ============================================================
print("🚀 Iniciando treino otimizado na RTX 5090...")
trainer.train()
